# JOCN 2024 QoT datasets: figures

Reads the netCDF datasets written by `generate_dataset.py` (one file per run and per
heuristic) and draws the figures of the dataset use case of the JOCN 2024 paper
(Section 5.C, Fig. 5) plus other views of the data. Only the dataset files are
needed: the topology, the paths and the scenario are embedded in each file.

Set `RUN_DIR` below to a run folder, or leave it as `None` to use the most recent
run under `results/JOCN_Benchmark_2024/generate_dataset/`. Figures are saved as
PNG, SVG and PDF to `<run>/figures/`.

In [ ]:
# Imports, run selection and plotting helpers.
from pathlib import Path
import json
import runpy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

HERE = Path.cwd()
if HERE.name != "JOCN_Benchmark_2024":
    HERE = next(p for p in [HERE, *HERE.parents] if (p / "examples" / "JOCN_Benchmark_2024").is_dir())
    HERE = HERE / "examples" / "JOCN_Benchmark_2024"

RUN_DIR = None  # e.g. Path("results/JOCN_Benchmark_2024/generate_dataset/20260927-140906")
if RUN_DIR is None:
    runs = sorted((HERE / "results" / "JOCN_Benchmark_2024" / "generate_dataset").glob("*/"))
    RUN_DIR = runs[-1]
RUN_DIR = Path(RUN_DIR)
FIGURES = RUN_DIR / "figures"
FIGURES.mkdir(exist_ok=True)

# One file per heuristic of the run, in the run's order; keys are the paper's heuristic names.
metadata = json.loads((RUN_DIR / "metadata.json").read_text(encoding="utf-8"))
datasets = {}
for file_name in metadata["files"]:
    ds = xr.open_dataset(RUN_DIR / file_name, engine="h5netcdf")
    datasets[ds.attrs["policy_paper_name"]] = ds
NAMES = list(datasets)
COLORS = dict(zip(NAMES, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))
print("run:", RUN_DIR)
print("heuristics:", NAMES)

plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})


def save(fig, stem):
    """Save a figure as PNG, SVG and PDF under the run's figures folder."""
    fig.tight_layout()
    for ext in ("png", "svg", "pdf"):
        fig.savefig(FIGURES / f"{stem}.{ext}", dpi=200)
    plt.show()

## Datasets at a glance

In [ ]:
# Blocking and sample counts of each heuristic, read from the file attributes.
summary = pd.DataFrame(
    {
        name: {
            "gym heuristic": ds.attrs["policy"],
            "arrivals": int(ds.attrs["arrivals_recorded"]),
            "accepted lightpaths": int(ds.attrs["lightpaths_accepted"]),
            "blocking ratio": float(ds.attrs["blocking_ratio"]),
            "co-propagating rows": ds.sizes.get("copropagating", 0),
            "load (Erlang)": float(ds.attrs["experiment_load"]),
            "launch power (dBm)": float(ds.attrs["experiment_launch_power_dbm"]),
            "seed": int(ds.attrs["experiment_seed"]),
            "gym commit": ds.attrs["gym_git_commit"][:10],
        }
        for name, ds in datasets.items()
    }
)
summary

## The topology object is rebuilt from the file

`load_topology` writes the embedded topology file to a temporary folder and loads
it with the same parameters the simulation used, giving back the gym's
`TopologyModel`.

In [ ]:
# Rebuild the gym TopologyModel from the first dataset and check it against the stored tables.
load_topology = runpy.run_path(str(HERE / "generate_dataset.py"))["load_topology"]
reference = datasets[NAMES[0]]
topology = load_topology(reference)
assert len(topology.links) == reference.sizes["link"]
assert len(topology.paths) == reference.sizes["path"]
print(f"{topology.topology_id}: {len(topology.node_names)} nodes, {len(topology.links)} links, "
      f"{sum(len(link.spans) for link in topology.links)} spans, {len(topology.paths)} paths")
config = json.loads(reference.attrs["scenario_config_json"])
{key: config[key] for key in ("k_paths", "num_spectrum_resources", "bit_rates", "load", "launch_power_dbm", "margin")}

## Navigating the relations in a dataset

The file is a small relational database. Every table is a dimension, and integer
variables point from one table to another:

| from | variable | points to |
|---|---|---|
| `lightpath` | `source`, `destination` | `node` |
| `lightpath` | `path_id` + `path_reversed` | `path` (route among the k shortest paths, read backwards when `path_reversed` is 1) |
| `lightpath x hop` | `hop_link` (hop 0 leaves the source; -1 beyond the route) | `link` |
| `lightpath` | `modulation_index` | `modulation` |
| `copropagating` | `copropagating_lightpath`, `copropagating_hop` | `lightpath`, `hop` |
| `copropagating` | `copropagating_modulation_index` | `modulation` |
| `path` | `path_nodes`, `path_links` (-1 padded) | `node`, `link` |
| `link` | `link_source`, `link_target` | `node` |
| `span` | `span_link` | `link` |

As in the gym, one path record serves both directions of a node pair, so `path_id`
identifies the route and `path_reversed` its direction; the per-hop variables
(`hop_*`, `copropagating_hop`) are already in traversal order, from the source.
Frequency slots convert to frequency with the `frequency_start_hz` and
`frequency_slot_bandwidth_hz` attributes. The cells below follow these relations
for one lightpath, one link and one node pair.

In [ ]:
# One lightpath: request, modulation format, route (node names) and QoT at establishment.
ds = datasets[NAMES[0]]
i = int(np.argmax(ds.hops.values))  # a long route, to make the example interesting
sample = ds.isel(lightpath=i)
nodes = ds.node.values
route_links = sample.hop_link.values[sample.hop_link.values >= 0]  # already from the source
route_nodes = ds.path_nodes.values[int(sample.path_id)]
route_nodes = route_nodes[route_nodes >= 0]
if int(sample.path_reversed):  # the stored path runs from the destination
    route_nodes = route_nodes[::-1]
print(f"lightpath {i}: request {int(sample.request_index)}, "
      f"{nodes[int(sample.source)]} -> {nodes[int(sample.destination)]}, {int(sample.bit_rate)} Gb/s")
print(f"  route: k={int(sample.path_k)}, {int(sample.hops)} links, {float(sample.route_length_km):.0f} km, "
      f"path {int(sample.path_id)} (reversed={int(sample.path_reversed)}): {' - '.join(nodes[route_nodes])}")
print(f"  {str(ds.modulation.values[int(sample.modulation_index)])}, slots {int(sample.slot_start)}"
      f"-{int(sample.slot_start) + int(sample.n_slots) - 1}, {float(sample.launch_power_dbm):.1f} dBm")
print(f"  GSNR {float(sample.gsnr_db):.2f} dB (ASE {float(sample.snr_ase_db):.2f} dB, "
      f"NLI {float(sample.snr_nli_db):.2f} dB), margin {float(sample.gsnr_margin_db):.2f} dB")

In [ ]:
# The same lightpath, hop by hop: each link with its endpoints, spans and noise contribution.
hops = pd.DataFrame(
    {
        "link": route_links,
        "from": nodes[route_nodes[:-1]],  # traversal direction (links themselves are undirected)
        "to": nodes[route_nodes[1:]],
        "length_km": ds.link_length_km.values[route_links],
        "spans": ds.link_n_spans.values[route_links],
        "ase_nsr": sample.hop_ase_nsr.values[: len(route_links)],
        "nli_nsr": sample.hop_nli_nsr.values[: len(route_links)],
        "occupancy_%": 100 * sample.hop_occupancy.values[: len(route_links)],
        "co-propagating": sample.hop_copropagating.values[: len(route_links)],
    }
)
hops["share_of_path_noise_%"] = 100 * (hops.ase_nsr + hops.nli_nsr) / (hops.ase_nsr + hops.nli_nsr).sum()
hops

In [ ]:
# Spans of the first link of the route (span table filtered by span_link).
first_link = int(route_links[0])
spans = np.flatnonzero(ds.span_link.values == first_link)
pd.DataFrame(
    {
        "length_km": ds.span_length_km.values[spans],
        "attenuation_dB_per_km": ds.span_attenuation_db_per_km.values[spans],
        "amplifier_NF_dB": ds.span_noise_figure_db.values[spans],
        "input_loss_dB": ds.span_input_loss_db.values[spans],
        "output_loss_dB": ds.span_output_loss_db.values[spans],
    },
    index=pd.Index(spans, name="span"),
)

In [ ]:
# Channels co-propagating with the lightpath on its first hop (ragged table), with frequency offsets.
rows = np.flatnonzero((ds.copropagating_lightpath.values == i) & (ds.copropagating_hop.values == 0))
slot_hz = ds.attrs["frequency_slot_bandwidth_hz"]
f0 = ds.attrs["frequency_start_hz"]
centre = f0 + slot_hz * (ds.copropagating_slot_start.values[rows] + ds.copropagating_n_slots.values[rows] / 2)
neighbours = pd.DataFrame(
    {
        "slot_start": ds.copropagating_slot_start.values[rows],
        "n_slots": ds.copropagating_n_slots.values[rows],
        "offset_GHz": (centre - float(sample.center_frequency_hz)) / 1e9,
        "bit_rate_Gbps": ds.copropagating_bit_rate.values[rows],
        "modulation": ds.modulation.values[ds.copropagating_modulation_index.values[rows]],
        "launch_power_dBm": ds.copropagating_launch_power_dbm.values[rows],
    }
).sort_values("offset_GHz", key=np.abs)
print(f"{len(neighbours)} channels share link {first_link} with lightpath {i}")
neighbours.head(10)

In [ ]:
# One link: every lightpath that crossed it, and how its QoT compares with the rest.
link = int(np.bincount(ds.hop_link.values[ds.hop_link.values >= 0]).argmax())  # busiest link
crosses = (ds.hop_link == link).any("hop")
print(f"link {link} ({nodes[int(ds.link_source[link])]} - {nodes[int(ds.link_target[link])]}): "
      f"{int(crosses.sum())} of {ds.sizes['lightpath']} lightpaths")
pd.DataFrame(
    {
        "through the link": ds.gsnr_db.where(crosses, drop=True).to_series().describe(),
        "elsewhere": ds.gsnr_db.where(~crosses, drop=True).to_series().describe(),
    }
).round(2)

In [ ]:
# One node pair: the k candidate paths in the path table and how often each was chosen.
pair = (int(sample.source), int(sample.destination))
candidates = np.flatnonzero(
    ((ds.path_source.values == pair[0]) & (ds.path_destination.values == pair[1]))
    | ((ds.path_source.values == pair[1]) & (ds.path_destination.values == pair[0]))
)
chosen = np.bincount(ds.path_id.values, minlength=ds.sizes["path"])
pd.DataFrame(
    {
        "rank": ds.path_rank.values[candidates],
        "hops": ds.path_hops.values[candidates],
        "length_km": ds.path_length_km.values[candidates].round(0),
        "times chosen": chosen[candidates],
        "route": [" - ".join(nodes[p[p >= 0]]) for p in ds.path_nodes.values[candidates]],
    },
    index=pd.Index(candidates, name="path_id"),
)

In [ ]:
# Group and export: mean GSNR per modulation format, and the lightpath table as a pandas DataFrame.
per_format = ds.gsnr_db.groupby(ds.modulation_index).mean().to_series()
per_format.index = ds.modulation.values[per_format.index]
print(per_format.round(2).to_string())
lightpath_table = ds[[name for name, var in ds.data_vars.items() if var.dims == ("lightpath",)]].to_dataframe()
lightpath_table.head()

## Fig. 5(a): GSNR of the accepted lightpaths

The paper notes that about 20% of the lightpaths of BM-LS-KSP exceed 20 dB, while
most of those of LB-BM-KSP stay below it. The dashed line marks 20 dB.

In [ ]:
# Fig. 5(a): GSNR distribution (histogram, normalised) of each heuristic.
fig, ax = plt.subplots(figsize=(5.5, 3.6))
bins = np.arange(5.0, 32.0, 0.5)
for name, ds in datasets.items():
    gsnr = ds.gsnr_db.values
    ax.hist(gsnr, bins=bins, density=True, histtype="step", lw=1.8, color=COLORS[name],
            label=f"{name} ({100 * np.mean(gsnr > 20):.0f}% > 20 dB)")
ax.axvline(20.0, color="0.4", ls="--", lw=1)
ax.set_xlabel("GSNR (dB)")
ax.set_ylabel("Probability density")
ax.legend(frameon=False, loc="upper left")
save(fig, "fig5a_gsnr_distribution_by_heuristic")

In [ ]:
# Fig. 5(a), cumulative view: CDF of the GSNR of each heuristic.
fig, ax = plt.subplots(figsize=(5.5, 3.6))
for name, ds in datasets.items():
    gsnr = np.sort(ds.gsnr_db.values)
    ax.plot(gsnr, np.arange(1, gsnr.size + 1) / gsnr.size, color=COLORS[name], lw=1.8, label=name)
ax.axvline(20.0, color="0.4", ls="--", lw=1)
ax.set_xlabel("GSNR (dB)")
ax.set_ylabel("Cumulative fraction of lightpaths")
ax.legend(frameon=False)
save(fig, "fig5a_gsnr_cdf_by_heuristic")

## Fig. 5(b): modulation-format usage

In [ ]:
# Fig. 5(b): share of accepted lightpaths per modulation format and heuristic.
formats = list(reference.modulation.values)
width = 0.8 / len(NAMES)
x = np.arange(len(formats))
fig, ax = plt.subplots(figsize=(5.5, 3.6))
for i, (name, ds) in enumerate(datasets.items()):
    share = np.bincount(ds.modulation_index.values, minlength=len(formats)) / ds.sizes["lightpath"]
    ax.bar(x + (i - (len(NAMES) - 1) / 2) * width, 100 * share, width, color=COLORS[name], label=name)
ax.set_xticks(x, formats)
ax.set_ylabel("Accepted lightpaths (%)")
ax.legend(frameon=False)
save(fig, "fig5b_modulation_format_usage_by_heuristic")

## GSNR versus route length

As in the margin-optimisation use case of the paper (Fig. 3(c)), the same modulation
format is used over a wide range of lengths: length alone is a poor predictor of the
QoT. Dashed lines are the GSNR thresholds of the formats.

In [ ]:
# GSNR vs route length for each heuristic (one panel per heuristic, subsampled for legibility).
fig, axes = plt.subplots(1, len(NAMES), figsize=(5.0 * len(NAMES), 3.6), sharey=True, squeeze=False)
rng = np.random.default_rng(0)
for ax, (name, ds) in zip(axes[0], datasets.items()):
    pick = rng.choice(ds.sizes["lightpath"], size=min(20_000, ds.sizes["lightpath"]), replace=False)
    ax.scatter(ds.route_length_km.values[pick], ds.gsnr_db.values[pick], s=2, alpha=0.25,
               color=COLORS[name], rasterized=True)
    for threshold in ds.modulation_gsnr_threshold_db.values:
        ax.axhline(threshold, color="0.6", ls="--", lw=0.8)
    ax.set_xlabel(f"Route length (km), {name}")
axes[0][0].set_ylabel("GSNR (dB)")
save(fig, "gsnr_vs_route_length_by_heuristic")

## Which impairment limits each lightpath

In [ ]:
# SNR from ASE only vs SNR from NLI only; points below the diagonal are NLI-limited.
fig, axes = plt.subplots(1, len(NAMES), figsize=(5.0 * len(NAMES), 3.8), sharex=True, sharey=True, squeeze=False)
for ax, (name, ds) in zip(axes[0], datasets.items()):
    ase, nli = ds.snr_ase_db.values, ds.snr_nli_db.values
    ax.hexbin(ase, nli, gridsize=60, bins="log", cmap="Blues", mincnt=1)
    low, high = np.nanmin([ase.min(), nli.min()]), np.nanmax([ase.max(), nli.max()])
    ax.plot([low, high], [low, high], color="0.3", lw=1)
    ax.set_xlabel(f"SNR, ASE only (dB), {name}")
    ax.text(0.03, 0.95, f"NLI-limited: {100 * np.mean(nli < ase):.1f}%", transform=ax.transAxes, va="top")
axes[0][0].set_ylabel("SNR, NLI only (dB)")
save(fig, "snr_ase_vs_snr_nli_by_heuristic")

In [ ]:
# Share of the per-link noise due to NLI versus the number of co-propagating channels on that link.
fig, ax = plt.subplots(figsize=(5.5, 3.6))
for name, ds in datasets.items():
    valid = ds.hop_link.values >= 0
    channels = ds.hop_copropagating.values[valid]
    share = (ds.hop_nli_nsr.values / (ds.hop_ase_nsr.values + ds.hop_nli_nsr.values))[valid]
    edges = np.arange(0, channels.max() + 6, 5)
    centers, means = [], []
    for low, high in zip(edges[:-1], edges[1:]):
        in_bin = (channels >= low) & (channels < high)
        if in_bin.sum() >= 50:
            centers.append((low + high) / 2)
            means.append(100 * share[in_bin].mean())
    ax.plot(centers, means, marker="o", ms=3, color=COLORS[name], label=name)
ax.set_xlabel("Co-propagating channels on the link")
ax.set_ylabel("NLI share of link noise (%)")
ax.legend(frameon=False)
save(fig, "nli_share_vs_copropagating_channels")

## How each heuristic uses the network

In [ ]:
# Rank of the chosen route among the k shortest paths (0 = shortest).
k = int(reference.attrs["topology_k_paths"])
width = 0.8 / len(NAMES)
fig, ax = plt.subplots(figsize=(5.5, 3.6))
for i, (name, ds) in enumerate(datasets.items()):
    share = np.bincount(ds.path_k.values, minlength=k) / ds.sizes["lightpath"]
    ax.bar(np.arange(k) + (i - (len(NAMES) - 1) / 2) * width, 100 * share, width, color=COLORS[name], label=name)
ax.set_xticks(np.arange(k), [str(i + 1) for i in range(k)])
ax.set_xlabel("Chosen route (k-th shortest path)")
ax.set_ylabel("Accepted lightpaths (%)")
ax.legend(frameon=False)
save(fig, "route_rank_by_heuristic")

In [ ]:
# GSNR margin above the threshold of the chosen modulation format.
fig, ax = plt.subplots(figsize=(5.5, 3.6))
bins = np.arange(0.0, 12.0, 0.25)
for name, ds in datasets.items():
    ax.hist(ds.gsnr_margin_db.values, bins=bins, density=True, histtype="step", lw=1.8,
            color=COLORS[name], label=f"{name} (median {np.median(ds.gsnr_margin_db.values):.2f} dB)")
ax.set_xlabel("GSNR margin over the modulation-format threshold (dB)")
ax.set_ylabel("Probability density")
ax.legend(frameon=False)
save(fig, "gsnr_margin_distribution_by_heuristic")

In [ ]:
# Spectrum occupancy of the traversed links at establishment.
fig, ax = plt.subplots(figsize=(5.5, 3.6))
bins = np.linspace(0.0, 1.0, 41)
for name, ds in datasets.items():
    occupancy = ds.hop_occupancy.values[ds.hop_link.values >= 0]
    ax.hist(100 * occupancy, bins=100 * bins, density=True, histtype="step", lw=1.8, color=COLORS[name], label=name)
ax.set_xlabel("Occupied slots on a traversed link (%)")
ax.set_ylabel("Probability density")
ax.legend(frameon=False)
save(fig, "link_occupancy_at_establishment_by_heuristic")

## Where the lightpaths go

Link usage drawn on the node coordinates embedded in the dataset: line width and
colour show how many accepted lightpaths crossed each link.

In [ ]:
# Topology map with the number of accepted lightpaths per link, one panel per heuristic.
fig, axes = plt.subplots(1, len(NAMES), figsize=(5.2 * len(NAMES), 4.6), squeeze=False)
for ax, (name, ds) in zip(axes[0], datasets.items()):
    links = ds.hop_link.values[ds.hop_link.values >= 0]
    usage = np.bincount(links, minlength=ds.sizes["link"])
    x, y = ds.node_x.values, ds.node_y.values
    scale = usage / usage.max()
    cmap = plt.get_cmap("Blues")
    for link in range(ds.sizes["link"]):
        s, t = int(ds.link_source.values[link]), int(ds.link_target.values[link])
        ax.plot([x[s], x[t]], [y[s], y[t]], color=cmap(0.25 + 0.75 * scale[link]), lw=0.8 + 5 * scale[link],
                solid_capstyle="round", zorder=1)
    ax.scatter(x, y, s=18, color="0.2", zorder=2)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ("left", "bottom"):
        ax.spines[side].set_visible(False)
    ax.set_xlabel(f"{name}: max {usage.max()} lightpaths on a link")
save(fig, "topology_link_usage_by_heuristic")